# 🚀 Qwen3.8-27B Server Launcher (2x Tesla T4)

### คำแนะนำก่อนเริ่ม:
1. ด้านขวา (Settings) -> **Accelerator**: เลือก **GPU T4 x 2**
2. ด้านขวา (Settings) -> **Internet**: ปรับเป็น **On**
3. กด **Run Cell 1** ด้านล่าง เพื่อติดตั้งและเริ่มรัน Server + Public URL ฟรี
4. เมื่อรันเสร็จ สามารถรัน Cell 2 เพื่อพูดคุย หรือใช้ Public URL ยิงจากภายนอกได้ทันที

In [ ]:
# ==============================================================================
# 🚀 Qwen3.8-27B All-in-One Server Launcher (2x Tesla T4)
# Features:
#   1. Auto-install llama.cpp CUDA 12
#   2. Direct stream download to /kaggle/tmp (Bypasses 19.5GB Kaggle storage limit)
#   3. Auto GPU Tensor Split (1:1 across 2x T4, 128k context, q4_0 KV cache)
#   4. Free Cloudflare Quick Tunnel (Instant Public HTTPS URL, zero token/domain needed)
# ==============================================================================

import os
import sys
import time
import re
import shutil
import subprocess
import urllib.request
from pathlib import Path

# --- Configuration ---
WORK_DIR = Path("/kaggle/tmp")
LOG_DIR = Path("/kaggle/working")
MODEL_PATH = WORK_DIR / "Qwen3.8-27B-UD-Q4_K_M.gguf"
MODEL_URL = "https://huggingface.co/unsloth/Qwen3.8-27B-GGUF/resolve/main/Qwen3.8-27B-UD-Q4_K_M.gguf"
LLAMA_DIR = WORK_DIR / "llama_cpp"
CF_BIN = WORK_DIR / "cloudflared"
LLAMA_LOG = LOG_DIR / "llama-server.log"
CF_LOG = LOG_DIR / "cloudflared.log"
PORT = 8080

WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# Helper: Clean old processes
# ------------------------------------------------------------------------------
def stop_services():
    print("🧹 Cleaning up old processes...")
    os.system("pkill -9 -f '[l]lama-server' >/dev/null 2>&1")
    os.system("pkill -9 -f '[c]loudflared' >/dev/null 2>&1")
    time.sleep(1)
    print("✅ Cleanup complete.")

stop_services()

# ------------------------------------------------------------------------------
# 1. Check GPU
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("🚀 Qwen3.8-27B All-in-One Server Launcher (2x Tesla T4)")
print("=" * 70)

try:
    gpu_out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True
    ).strip().splitlines()
    print(f"🎮 Detected {len(gpu_out)} GPU(s):")
    for idx, g in enumerate(gpu_out):
        print(f"   [{idx}] {g}")
    if len(gpu_out) < 2:
        print("⚠️ Warning: Less than 2 GPUs detected. Set Settings -> Accelerator -> GPU T4 x 2")
except Exception as e:
    print(f"⚠️ Could not query nvidia-smi: {e}")

# ------------------------------------------------------------------------------
# 2. Download llama.cpp CUDA binaries
# ------------------------------------------------------------------------------
print("\n⚡ [1/4] Checking llama.cpp CUDA 12 binaries...")
server_bin = None
for candidate in WORK_DIR.rglob("llama-server"):
    if candidate.is_file() and os.access(candidate, os.X_OK):
        server_bin = str(candidate)
        break

if not server_bin:
    tar_path = WORK_DIR / "llama_cpp_binaries.tar.gz"
    bin_url = "https://github.com/ai-dock/llama.cpp-cuda/releases/download/b9628/llama.cpp-b9628-cuda-12.8-amd64.tar.gz"
    if not tar_path.exists():
        print(f"📥 Downloading prebuilt llama.cpp CUDA from GitHub...")
        subprocess.run(["wget", "-q", "--show-progress", bin_url, "-O", str(tar_path)], check=True)
    
    print("📦 Extracting binaries to /kaggle/tmp/llama_cpp...")
    LLAMA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xzf", str(tar_path), "-C", str(LLAMA_DIR)], check=True)
    for candidate in WORK_DIR.rglob("llama-server"):
        if candidate.is_file() and os.access(candidate, os.X_OK):
            server_bin = str(candidate)
            break

extract_dir = os.path.dirname(server_bin)
os.environ["PATH"] = f"{extract_dir}:{os.environ.get('PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = f"{extract_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
print(f"✅ llama-server ready: {server_bin}")

# ------------------------------------------------------------------------------
# 3. Download cloudflared
# ------------------------------------------------------------------------------
print("\n☁️ [2/4] Checking cloudflared binary...")
if not CF_BIN.exists():
    print("📥 Downloading cloudflared Linux binary...")
    cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    subprocess.run(["wget", "-q", "--show-progress", cf_url, "-O", str(CF_BIN)], check=True)
    CF_BIN.chmod(0o755)
print(f"✅ cloudflared ready: {CF_BIN}")

# ------------------------------------------------------------------------------
# 4. Check mounted Dataset in /kaggle/input, else stream download (~17 GB)
# ------------------------------------------------------------------------------
print(f"\n📥 [3/4] Checking model file: {MODEL_PATH.name}...")

mounted_models = [
    p for p in Path("/kaggle/input").rglob("*.gguf")
    if "mmproj" not in p.name.lower()
] if Path("/kaggle/input").exists() else []

if mounted_models:
    MODEL_PATH = mounted_models[0]
    size_gb = MODEL_PATH.stat().st_size / (1024**3)
    print(f"⚡ Found mounted Dataset: {MODEL_PATH.name} ({size_gb:.2f} GB) [Instant Mount 0s]")
else:
    if not MODEL_PATH.exists():
        print("🚀 Downloading Qwen3.8-27B-UD-Q4_K_M.gguf (~17 GB) from Hugging Face...")
        print("   (Takes about 1.5 - 2 minutes on Kaggle high-speed connection)")
        subprocess.run([
            "wget", "--continue", "--progress=bar:force:noscroll", "--show-progress",
            MODEL_URL, "-O", str(MODEL_PATH)
        ], check=True)
    size_gb = MODEL_PATH.stat().st_size / (1024**3)
    print(f"✅ Model ready ({size_gb:.2f} GB)")

# ------------------------------------------------------------------------------
# 5. Launch llama-server in background
# ------------------------------------------------------------------------------
print(f"\n🚀 [4/4] Starting llama-server on port {PORT}...")

help_text = subprocess.run([server_bin, "--help"], capture_output=True, text=True, env=os.environ).stdout or ""
server_cmd = [
    server_bin,
    "-m", str(MODEL_PATH),
    "--alias", "qwen3.8-27b",
    "-ngl", "99",
    "-sm", "layer",
    "-ts", "1,1",
    "-c", "131072",
    "-np", "1",
    "-fa", "on",
    "-ctk", "q4_0",
    "-ctv", "q4_0",
    "-b", "2048",
    "-ub", "512",
    "--host", "127.0.0.1",
    "--port", str(PORT),
]

if "-fit" in help_text or "--fit" in help_text:
    server_cmd.extend(["-fit", "off"])

llama_log_handle = open(LLAMA_LOG, "w", encoding="utf-8", buffering=1)
server_proc = subprocess.Popen(
    server_cmd,
    stdout=llama_log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ,
    start_new_session=True
)
print(f"   llama-server background PID: {server_proc.pid}")
print(f"   Log: {LLAMA_LOG}")

# Wait for local health check
print("⏳ Loading model weights into 2x Tesla T4 VRAM (Context: 128k)...")
local_ready = False
start_time = time.time()
while time.time() - start_time < 300:
    if server_proc.poll() is not None:
        print("\n❌ llama-server crashed! Last logs:")
        print("\n".join(Path(LLAMA_LOG).read_text(errors="replace").splitlines()[-30:]))
        raise RuntimeError("llama-server failed to start")
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/models")
        with urllib.request.urlopen(req, timeout=3) as resp:
            if resp.status == 200:
                local_ready = True
                break
    except Exception:
        pass
    time.sleep(2)

if not local_ready:
    last_log = "\n".join(Path(LLAMA_LOG).read_text(errors="replace").splitlines()[-30:])
    raise TimeoutError(f"Timed out waiting for llama-server to initialize.\nLast logs:\n{last_log}")
print("✅ Local llama-server is READY!")

# ------------------------------------------------------------------------------
# 6. Start Cloudflare Quick Tunnel (Free Public HTTPS URL)
# ------------------------------------------------------------------------------
print("☁️ Launching Cloudflare Quick Tunnel...")
cf_log_handle = open(CF_LOG, "w", encoding="utf-8", buffering=1)
cf_proc = subprocess.Popen(
    [str(CF_BIN), "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=cf_log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 45:
    if cf_proc.poll() is not None:
        print("\n❌ cloudflared crashed! Last logs:")
        print("\n".join(Path(CF_LOG).read_text(errors="replace").splitlines()[-30:]))
        break
    if CF_LOG.exists():
        content = CF_LOG.read_text(errors="replace")
        match = re.search(r"https://[-0-9a-z]+\.trycloudflare\.com", content)
        if match:
            public_url = match.group(0)
            break
    time.sleep(2)

print("\n" + "=" * 70)
print("🎉 QWEN 3.8 (27B) SERVER IS LIVE!")
print("=" * 70)
print(f"🖥️ Local Base URL  : http://127.0.0.1:{PORT}/v1")
if public_url:
    print(f"🌐 Free Public URL : {public_url}/v1")
else:
    print(f"⚠️ Tunnel URL pending... Check log: !cat {CF_LOG}")
print("🧠 Model Alias     : qwen3.8-27b")
print("📏 Context Length  : 131,072 tokens (128k)")
print("⚡ KV Cache Quant  : q4_0 / q4_0")
print("=" * 70)
print("\n✅ Cell นี้ทำงานเสร็จสิ้นแล้ว! Server ยังคงรันอยู่เบื้องหลัง")
print("👉 คุณสามารถรัน Cell ถัดไปเพื่อทดสอบสนทนา หรือยิงจากภายนอกผ่าน Public URL ได้ทันที")


In [ ]:
# ==============================================================================
# 💬 Cell 2: ทดสอบพูดคุยกับ Qwen3.8-27B (ส่งคำถามได้ทันทีใน Notebook)
# ==============================================================================
import json
import urllib.request

def ask_qwen(prompt, max_tokens=512):
    url = 'http://127.0.0.1:8080/v1/chat/completions'
    payload = {
        'model': 'qwen3.8-27b',
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': max_tokens,
        'temperature': 0.7
    }
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'}
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        res = json.loads(resp.read().decode('utf-8'))
        return res['choices'][0]['message']['content']

# ทดสอบส่งคำถาม
prompt = 'สวัสดีครับ ช่วยแนะนำตัวสั้นๆ และบอกว่าคุณคือโมเดลอะไร มีจุดเด่นอะไรบ้าง'
print(f'👤 User: {prompt}\n')
print('🤖 Qwen กำลังตอบ...')
reply = ask_qwen(prompt)
print(f'\n{reply}')


In [ ]:
# ==============================================================================
# 🛑 Cell 3: คำสั่งปิด Server ทั้งหมด (รันเมื่อต้องการหยุดการทำงาน)
# ==============================================================================
stop_services()
